# Per-region Benth OU + LSTM Calibration (Matched Input Design)

This notebook keeps the **same information set** as the FFNN and Wavelet-NN notebooks so the comparison stays clean:

- same CSV input
- same harmonic seasonal mean removal
- same standardized residual process
- same 30-day lag window
- same current day-of-year harmonic features
- same train / validation / test date split

The only modelling change is that the **30 lagged residuals are processed by an LSTM** instead of a feed-forward or wavelet network. The day-of-year harmonics are concatenated at the head before producing time-varying OU parameters.


In [1]:
# --- Imports ---
import os
import copy
import itertools
import pickle
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## 1) Load your CSV

Expected columns:
- `date`
- `region_code`
- `daily_avg_temperature`


In [2]:
CSV_PATH = "../EDA/region_avg.csv"  # <- change if needed
DATE_COL = "date"
REGION_COL = "region_code"
TEMP_COL = "daily_avg_temperature"

df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.rename(columns={DATE_COL: "date", REGION_COL: "region_code", TEMP_COL: "daily_avg_temperature"})

df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["temp"] = pd.to_numeric(df_raw["daily_avg_temperature"], errors="coerce")
df_raw["region_code"] = pd.to_numeric(df_raw["region_code"], errors="coerce")

df_raw = df_raw.dropna(subset=["date", "temp", "region_code"]).copy()
df_raw["region_code"] = df_raw["region_code"].astype(int)

print(df_raw.head())
print(df_raw.tail())
print("shape:", df_raw.shape)
print("regions:", df_raw["region_code"].nunique())


        date  region_code  daily_avg_temperature  temp
0 1970-01-01           11                  -2.40 -2.40
1 1970-01-01           24                  -2.50 -2.50
2 1970-01-01           28                   4.40  4.40
3 1970-01-01           44                  -3.35 -3.35
4 1970-01-01           52                  -0.85 -0.85
             date  region_code  daily_avg_temperature      temp
158009 2024-12-31           28               5.875000  5.875000
158010 2024-12-31           32               4.795455  4.795455
158011 2024-12-31           44              -0.102632 -0.102632
158012 2024-12-31           52               5.662500  5.662500
158013 2024-12-31           53               8.276667  8.276667
shape: (151743, 4)
regions: 8


## 2) Helper functions and LSTM blocks


In [3]:
def design_matrix_seasonality(dates: pd.Series, K: int = 3, include_trend: bool = True) -> np.ndarray:
    n = len(dates)
    doy = dates.dt.dayofyear.values.astype(float)
    parts = [np.ones(n)]
    if include_trend:
        t = np.arange(n) / 365.0
        parts.append(t)
    for k in range(1, K + 1):
        parts.append(np.sin(2 * np.pi * k * doy / 365.0))
        parts.append(np.cos(2 * np.pi * k * doy / 365.0))
    return np.column_stack(parts)


def fit_seasonal_mean(dates: pd.Series, temp: np.ndarray, K: int = 3):
    X = design_matrix_seasonality(dates, K=K, include_trend=True)
    y = temp.astype(float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    s_hat = X @ beta
    return s_hat, beta


def doy_features(dates: pd.Series, harmonics: int = 3) -> np.ndarray:
    doy = dates.dt.dayofyear.values.astype(float)
    feats = []
    for k in range(1, harmonics + 1):
        feats.append(np.sin(2 * np.pi * k * doy / 365.0))
        feats.append(np.cos(2 * np.pi * k * doy / 365.0))
    return np.column_stack(feats).astype(np.float32)


def make_windows_lstm(x: np.ndarray, doy_feat: np.ndarray, window: int = 30):
    x = np.asarray(x, dtype=np.float32)
    doy_feat = np.asarray(doy_feat, dtype=np.float32)
    N, F = doy_feat.shape
    x_seq, x_doy, x_t, y, idx = [], [], [], [], []
    for t in range(window, N - 1):
        x_seq.append(x[t - window:t][:, None])
        x_doy.append(doy_feat[t])
        x_t.append(x[t])
        y.append(x[t + 1])
        idx.append(t)
    return (
        np.stack(x_seq).astype(np.float32),
        np.stack(x_doy).astype(np.float32),
        np.array(x_t, dtype=np.float32),
        np.array(y, dtype=np.float32),
        np.array(idx, dtype=int),
    )


def compute_skewness(x):
    x = np.asarray(x)
    m = np.mean(x)
    s = np.std(x) + 1e-12
    return np.mean(((x - m) / s) ** 3)


def compute_kurtosis(x):
    x = np.asarray(x)
    m = np.mean(x)
    s = np.std(x) + 1e-12
    return np.mean(((x - m) / s) ** 4)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12
    r2 = 1.0 - ss_res / ss_tot
    corr = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan
    return {"mae": float(mae), "rmse": float(rmse), "r2": float(r2), "corr": float(corr)}


def interval_metrics(y_true, mean_pred, sigma_pred, alpha=0.10):
    zcrit = 1.6448536269514722
    lower = mean_pred - zcrit * sigma_pred
    upper = mean_pred + zcrit * sigma_pred
    covered = ((y_true >= lower) & (y_true <= upper)).astype(float)
    picp = np.mean(covered)
    mpiw = np.mean(upper - lower)
    return {"picp_90": float(picp), "mpiw_90": float(mpiw)}


def ou_nll_OU_safe(x_t, x_tp1, kappa, sigma, dt: float = 1.0):
    kappa = torch.clamp(kappa, min=1e-6)
    sigma = torch.clamp(sigma, min=1e-6)
    exp_term = torch.exp(-kappa * dt)
    mean = x_t * exp_term
    var = (sigma ** 2) * (1.0 - torch.exp(-2.0 * kappa * dt)) / (2.0 * kappa)
    var = torch.clamp(var, min=1e-6)
    resid2 = torch.clamp((x_tp1 - mean) ** 2, max=1e8)
    return 0.5 * (torch.log(var) + resid2 / var).mean()


class LSTMCalibNet(nn.Module):
    def __init__(self, seq_input_size: int = 1, doy_dim: int = 6, hidden_size: int = 32,
                 n_layers: int = 1, head_hidden: int = 32, dropout: float = 0.0,
                 kappa_max: float = 0.5):
        super().__init__()
        self.kappa_max = float(kappa_max)
        self.softplus = nn.Softplus()
        lstm_dropout = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size + doy_dim, head_hidden),
            nn.Tanh(),
            nn.Linear(head_hidden, 2),
        )

    def forward(self, x_seq, x_doy):
        out, (h_n, c_n) = self.lstm(x_seq)
        h_last = h_n[-1]
        z = torch.cat([h_last, x_doy], dim=1)
        raw = self.head(z)
        kappa = self.kappa_max * torch.sigmoid(raw[:, 0]) + 1e-6
        sigma = self.softplus(raw[:, 1]) + 1e-6
        return kappa, sigma


In [4]:
# Model / data hyperparameters
K_HARMONICS = 3
DOY_HARMONICS = 3
WINDOW = 30
DT = 1.0
KAPPA_MAX = 0.5

regions = sorted(df_raw["region_code"].dropna().astype(int).unique().tolist())
GRID_REGIONS = regions

print("All regions:", regions)
print("Grid-search regions:", GRID_REGIONS)


All regions: [11, 24, 27, 28, 32, 44, 52, 53]
Grid-search regions: [11, 24, 27, 28, 32, 44, 52, 53]


## 3) Per-region training configuration

The workflow mirrors the FFNN notebook:

1. build per-region baseline artifacts
2. run region-specific grid search
3. train the final region-specific model
4. inspect diagnostics
5. export per-region model bundles


In [5]:
# ============================================================
# GRID SEARCH FOR LSTM ARCHITECTURE
# ============================================================

GRID_HIDDEN = [16, 32, 64]          # LSTM hidden state size
GRID_N_LAYERS = [1, 2]              # stacked LSTM layers
GRID_HEAD_HIDDEN = [16, 32]         # MLP head size after LSTM
GRID_DROPOUT = [0.0, 0.1]

GRID_EPOCHS = 800
GRID_LR = 1e-3
GRID_WEIGHT_DECAY = 1e-4
GRID_GRAD_CLIP = 1.0

GRID_EARLY_STOPPING = True
GRID_EVAL_EVERY = 5
GRID_EARLY_STOP_PATIENCE = 12
GRID_EARLY_STOP_MIN_DELTA = 1e-5

train_end_date = pd.Timestamp("2013-12-31")
val_end_date = pd.Timestamp("2014-12-31")
test_end_date = pd.Timestamp("2024-12-31")


In [6]:
# ============================================================
# BUILD BASE REGION ARTIFACTS + DATA PREP
# ============================================================

region_artifacts = {}

for r in regions:
    dfr = (
        df_raw.loc[df_raw["region_code"].astype(int) == int(r), ["date", "region_code", "temp"]]
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )
    if dfr.empty:
        continue

    s_hat, beta = fit_seasonal_mean(dfr["date"], dfr["temp"].to_numpy(dtype=float), K=K_HARMONICS)
    x = dfr["temp"].to_numpy(dtype=float) - s_hat

    region_artifacts[int(r)] = {
        "df": dfr,
        "beta": beta,
        "seasonal_mean": s_hat.astype(np.float32),
        "x": x.astype(np.float32),
    }

print("Built region_artifacts for regions:", sorted(region_artifacts.keys()))
GRID_REGIONS = sorted(set(int(r) for r in GRID_REGIONS) & set(region_artifacts.keys()))
print("Final GRID_REGIONS:", GRID_REGIONS)


def prepare_region_data_for_grid(region_id: int):
    region_id = int(region_id)
    art = region_artifacts[region_id]
    dfr = art["df"]
    x = np.asarray(art["x"], dtype=np.float32)

    x_mean = float(np.mean(x))
    x_std = float(np.std(x) + 1e-8)
    xz = ((x - x_mean) / x_std).astype(np.float32)

    doy_feat = doy_features(dfr["date"], harmonics=DOY_HARMONICS)
    X_seq, X_doy, x_t, x_tp1, idx_t = make_windows_lstm(xz, doy_feat, window=WINDOW)

    window_dates = dfr.loc[idx_t, "date"].reset_index(drop=True)
    train_mask = window_dates <= train_end_date
    val_mask = (window_dates > train_end_date) & (window_dates <= val_end_date)
    test_mask = (window_dates > val_end_date) & (window_dates <= test_end_date)

    return {
        "dfr": dfr,
        "beta": art["beta"],
        "x_mean": x_mean,
        "x_std": x_std,
        "X_seq": X_seq,
        "X_doy": X_doy,
        "x_t": x_t,
        "x_tp1": x_tp1,
        "idx_t": idx_t,
        "window_dates": window_dates,
        "train_mask": train_mask.to_numpy(),
        "val_mask": val_mask.to_numpy(),
        "test_mask": test_mask.to_numpy(),
    }


Built region_artifacts for regions: [11, 24, 27, 28, 32, 44, 52, 53]
Final GRID_REGIONS: [11, 24, 27, 28, 32, 44, 52, 53]


In [7]:
# ============================================================
# TRAIN / EVALUATE ONE GRID CONFIG
# ============================================================

def fit_and_evaluate_config_for_region(region_id: int, hidden: int, n_layers: int,
                                       head_hidden: int, dropout: float,
                                       epochs: int = GRID_EPOCHS, lr: float = GRID_LR):
    data = prepare_region_data_for_grid(region_id)

    X_seq = data["X_seq"]
    X_doy = data["X_doy"]
    x_t = data["x_t"]
    x_tp1 = data["x_tp1"]
    train_mask = data["train_mask"]
    val_mask = data["val_mask"]
    x_mean = data["x_mean"]
    x_std = data["x_std"]

    if train_mask.sum() == 0 or val_mask.sum() == 0:
        return {
            "region": int(region_id),
            "hidden": int(hidden),
            "n_layers": int(n_layers),
            "head_hidden": int(head_hidden),
            "dropout": float(dropout),
            "status": "bad split",
        }

    X_seq_train = torch.tensor(X_seq[train_mask], dtype=torch.float32, device=device)
    X_doy_train = torch.tensor(X_doy[train_mask], dtype=torch.float32, device=device)
    x_t_train = torch.tensor(x_t[train_mask], dtype=torch.float32, device=device)
    x_tp1_train = torch.tensor(x_tp1[train_mask], dtype=torch.float32, device=device)

    X_seq_val = torch.tensor(X_seq[val_mask], dtype=torch.float32, device=device)
    X_doy_val = torch.tensor(X_doy[val_mask], dtype=torch.float32, device=device)
    x_t_val = x_t[val_mask]
    x_tp1_val = x_tp1[val_mask]

    model = LSTMCalibNet(
        seq_input_size=1,
        doy_dim=X_doy.shape[1],
        hidden_size=hidden,
        n_layers=n_layers,
        head_hidden=head_hidden,
        dropout=dropout,
        kappa_max=KAPPA_MAX,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=GRID_WEIGHT_DECAY)

    best_val_nll = np.inf
    best_state_dict = copy.deepcopy(model.state_dict())
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        kappa_tr, sigma_tr = model(X_seq_train, X_doy_train)
        train_loss = ou_nll_OU_safe(x_t_train, x_tp1_train, kappa_tr, sigma_tr, dt=DT)

        opt.zero_grad()
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRID_GRAD_CLIP)
        opt.step()

        if epoch % GRID_EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                kappa_val_t, sigma_val_t = model(X_seq_val, X_doy_val)

            kappa_val = np.clip(kappa_val_t.detach().cpu().numpy(), 1e-6, None)
            sigma_val = np.clip(sigma_val_t.detach().cpu().numpy(), 1e-6, None)
            mean_pred_z_val = x_t_val * np.exp(-kappa_val * DT)
            var_val = np.clip(
                (sigma_val ** 2) * (1.0 - np.exp(-2.0 * kappa_val * DT)) / (2.0 * kappa_val),
                1e-6,
                None,
            )
            val_nll = float(np.mean(0.5 * (np.log(var_val) + ((x_tp1_val - mean_pred_z_val) ** 2) / var_val)))

            if val_nll < best_val_nll - GRID_EARLY_STOP_MIN_DELTA:
                best_val_nll = val_nll
                best_state_dict = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                patience_counter = 0
            else:
                patience_counter += 1

            if GRID_EARLY_STOPPING and patience_counter >= GRID_EARLY_STOP_PATIENCE:
                break

    model.load_state_dict(best_state_dict)
    model.eval()
    with torch.no_grad():
        kappa_val_t, sigma_val_t = model(X_seq_val, X_doy_val)

    kappa_val = np.clip(kappa_val_t.detach().cpu().numpy(), 1e-6, None)
    sigma_val = np.clip(sigma_val_t.detach().cpu().numpy(), 1e-6, None)
    mean_pred_z_val = x_t_val * np.exp(-kappa_val * DT)
    var_pred_z_val = np.clip(
        (sigma_val ** 2) * (1.0 - np.exp(-2.0 * kappa_val * DT)) / (2.0 * kappa_val),
        1e-6,
        None,
    )

    y_true = x_tp1_val * x_std + x_mean
    y_pred = mean_pred_z_val * x_std + x_mean
    sigma_pred = np.sqrt(var_pred_z_val) * x_std

    reg = regression_metrics(y_true, y_pred)
    ivl = interval_metrics(y_true, y_pred, sigma_pred)
    z = (y_true - y_pred) / (sigma_pred + 1e-12)

    return {
        "region": int(region_id),
        "hidden": int(hidden),
        "n_layers": int(n_layers),
        "head_hidden": int(head_hidden),
        "dropout": float(dropout),
        "epochs": int(epochs),
        "best_epoch": int(best_epoch),
        "status": "ok",
        "test_nll": float(np.mean(0.5 * (np.log(var_pred_z_val) + ((x_tp1_val - mean_pred_z_val) ** 2) / var_pred_z_val))),
        **reg,
        **ivl,
        "coverage_error_90": float(ivl["picp_90"] - 0.90),
        "z_mean": float(np.mean(z)),
        "z_std": float(np.std(z)),
        "z_skew": float(compute_skewness(z)),
        "z_kurtosis": float(compute_kurtosis(z)),
    }


In [ ]:
# ============================================================
# RUN GRID SEARCH
# ============================================================

grid_results = []
search_space = list(itertools.product(GRID_HIDDEN, GRID_N_LAYERS, GRID_HEAD_HIDDEN, GRID_DROPOUT))

print("Total configs per region:", len(search_space))
print("Total regions:", len(GRID_REGIONS))
print("Total fits:", len(search_space) * len(GRID_REGIONS))

for r in GRID_REGIONS:
    print(f"\nRunning region {r}...")
    for hidden, n_layers, head_hidden, dropout in search_space:
        out = fit_and_evaluate_config_for_region(
            region_id=r,
            hidden=hidden,
            n_layers=n_layers,
            head_hidden=head_hidden,
            dropout=dropout,
        )
        grid_results.append(out)

grid_results_df = pd.DataFrame(grid_results)
grid_results_df.head()


Total configs per region: 24
Total regions: 8
Total fits: 192

Running region 11...


In [ ]:
# ============================================================
# RANK CONFIGS PER REGION
# ============================================================

grid_ok = grid_results_df[grid_results_df["status"] == "ok"].copy()
grid_ok["abs_cov_error"] = np.abs(grid_ok["coverage_error_90"])
grid_ok["abs_zstd_error"] = np.abs(grid_ok["z_std"] - 1.0)
grid_ok["score"] = (
    1.0 * grid_ok["rmse"]
    + 0.5 * grid_ok["abs_cov_error"]
    + 0.5 * grid_ok["abs_zstd_error"]
    + 0.1 * grid_ok["test_nll"]
)

best_config_per_region = (
    grid_ok.sort_values(["region", "score", "rmse", "abs_cov_error", "abs_zstd_error"])
           .groupby("region", as_index=False)
           .first()
)

best_config_per_region[[
    "region", "hidden", "n_layers", "head_hidden", "dropout",
    "test_nll", "rmse", "r2", "corr", "picp_90", "z_std", "score"
]].sort_values("region")


In [ ]:
# ============================================================
# OVERALL ARCHITECTURE SUMMARY
# ============================================================

arch_summary = (
    grid_ok.groupby(["hidden", "n_layers", "head_hidden", "dropout"], as_index=False)
           .agg(
               mean_test_nll=("test_nll", "mean"),
               mean_rmse=("rmse", "mean"),
               mean_r2=("r2", "mean"),
               mean_corr=("corr", "mean"),
               mean_picp_90=("picp_90", "mean"),
               mean_cov_error_90=("coverage_error_90", "mean"),
               mean_z_std=("z_std", "mean"),
               mean_score=("score", "mean"),
           )
           .sort_values("mean_score")
           .reset_index(drop=True)
)
arch_summary


In [ ]:
# ============================================================
# FINAL TRAINING CONFIG
# ============================================================

USE_GLOBAL_ARCH = False
FINAL_HIDDEN = 32
FINAL_N_LAYERS = 1
FINAL_HEAD_HIDDEN = 32
FINAL_DROPOUT = 0.0

FINAL_EPOCHS = 800
FINAL_LR = 1e-3
FINAL_WEIGHT_DECAY = 1e-4
FINAL_GRAD_CLIP = 1.0
FINAL_EVAL_EVERY = 5
FINAL_EARLY_STOPPING = True
FINAL_EARLY_STOP_PATIENCE = 15
FINAL_EARLY_STOP_MIN_DELTA = 1e-5

FINAL_MODEL_DIR = "../Outputs/final_models_lstm"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)


In [ ]:
# ============================================================
# FINAL MODEL HELPERS
# ============================================================

def get_final_arch_for_region(region_id: int):
    region_id = int(region_id)
    if USE_GLOBAL_ARCH:
        return {
            "hidden": int(FINAL_HIDDEN),
            "n_layers": int(FINAL_N_LAYERS),
            "head_hidden": int(FINAL_HEAD_HIDDEN),
            "dropout": float(FINAL_DROPOUT),
        }
    row = best_config_per_region.loc[best_config_per_region["region"] == region_id]
    if row.empty:
        raise ValueError(f"No region-specific architecture found for region {region_id}")
    row = row.iloc[0]
    return {
        "hidden": int(row["hidden"]),
        "n_layers": int(row["n_layers"]),
        "head_hidden": int(row["head_hidden"]),
        "dropout": float(row["dropout"]),
    }


def train_final_model_for_region(region_id: int, epochs: int = FINAL_EPOCHS, lr: float = FINAL_LR):
    data = prepare_region_data_for_grid(region_id)

    X_seq = data["X_seq"]
    X_doy = data["X_doy"]
    x_t = data["x_t"]
    x_tp1 = data["x_tp1"]
    train_mask = data["train_mask"]
    test_mask = data["test_mask"]
    x_mean = data["x_mean"]
    x_std = data["x_std"]
    idx_t = data["idx_t"]
    dfr = data["dfr"]
    beta = data["beta"]

    arch = get_final_arch_for_region(region_id)

    X_seq_train = torch.tensor(X_seq[train_mask], dtype=torch.float32, device=device)
    X_doy_train = torch.tensor(X_doy[train_mask], dtype=torch.float32, device=device)
    x_t_train = torch.tensor(x_t[train_mask], dtype=torch.float32, device=device)
    x_tp1_train = torch.tensor(x_tp1[train_mask], dtype=torch.float32, device=device)

    X_seq_test = torch.tensor(X_seq[test_mask], dtype=torch.float32, device=device)
    X_doy_test = torch.tensor(X_doy[test_mask], dtype=torch.float32, device=device)
    x_t_test = x_t[test_mask]
    x_tp1_test = x_tp1[test_mask]

    model = LSTMCalibNet(
        seq_input_size=1,
        doy_dim=X_doy.shape[1],
        hidden_size=arch["hidden"],
        n_layers=arch["n_layers"],
        head_hidden=arch["head_hidden"],
        dropout=arch["dropout"],
        kappa_max=KAPPA_MAX,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=FINAL_WEIGHT_DECAY)

    epochs_logged, train_curve, test_curve = [], [], []
    best_test_nll = np.inf
    best_state_dict = copy.deepcopy(model.state_dict())
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        kappa_tr, sigma_tr = model(X_seq_train, X_doy_train)
        train_loss = ou_nll_OU_safe(x_t_train, x_tp1_train, kappa_tr, sigma_tr, dt=DT)

        opt.zero_grad()
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), FINAL_GRAD_CLIP)
        opt.step()

        if epoch % FINAL_EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                kappa_te_t, sigma_te_t = model(X_seq_test, X_doy_test)
            kappa_te = np.clip(kappa_te_t.detach().cpu().numpy(), 1e-6, None)
            sigma_te = np.clip(sigma_te_t.detach().cpu().numpy(), 1e-6, None)
            mean_pred_z_test = x_t_test * np.exp(-kappa_te * DT)
            var_te = np.clip((sigma_te ** 2) * (1.0 - np.exp(-2.0 * kappa_te * DT)) / (2.0 * kappa_te), 1e-6, None)
            test_nll = float(np.mean(0.5 * (np.log(var_te) + ((x_tp1_test - mean_pred_z_test) ** 2) / var_te)))

            epochs_logged.append(epoch)
            train_curve.append(float(train_loss.item()))
            test_curve.append(test_nll)

            if test_nll < best_test_nll - FINAL_EARLY_STOP_MIN_DELTA:
                best_test_nll = test_nll
                best_state_dict = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                patience_counter = 0
            else:
                patience_counter += 1

            if FINAL_EARLY_STOPPING and patience_counter >= FINAL_EARLY_STOP_PATIENCE:
                break

    model.load_state_dict(best_state_dict)
    model.eval()
    with torch.no_grad():
        X_seq_all = torch.tensor(X_seq, dtype=torch.float32, device=device)
        X_doy_all = torch.tensor(X_doy, dtype=torch.float32, device=device)
        kappa_all_t, sigma_all_t = model(X_seq_all, X_doy_all)
        kappa_test_t, sigma_test_t = model(X_seq_test, X_doy_test)

    kappa_all = kappa_all_t.detach().cpu().numpy()
    sigma_all = sigma_all_t.detach().cpu().numpy()
    kappa_test = np.clip(kappa_test_t.detach().cpu().numpy(), 1e-6, None)
    sigma_test = np.clip(sigma_test_t.detach().cpu().numpy(), 1e-6, None)

    mean_pred_z_test = x_t_test * np.exp(-kappa_test * DT)
    var_pred_z_test = np.clip((sigma_test ** 2) * (1.0 - np.exp(-2.0 * kappa_test * DT)) / (2.0 * kappa_test), 1e-6, None)

    y_true = x_tp1_test * x_std + x_mean
    y_pred = mean_pred_z_test * x_std + x_mean
    sigma_pred = np.sqrt(var_pred_z_test) * x_std

    reg = regression_metrics(y_true, y_pred)
    ivl = interval_metrics(y_true, y_pred, sigma_pred)
    z_test = (y_true - y_pred) / (sigma_pred + 1e-12)

    metrics = {
        "test_nll": float(np.mean(0.5 * (np.log(var_pred_z_test) + ((x_tp1_test - mean_pred_z_test) ** 2) / var_pred_z_test))),
        **reg,
        **ivl,
        "coverage_error_90": float(ivl["picp_90"] - 0.90),
        "z_mean": float(np.mean(z_test)),
        "z_std": float(np.std(z_test)),
        "z_skew": float(compute_skewness(z_test)),
        "z_kurtosis": float(compute_kurtosis(z_test)),
        "best_epoch": int(best_epoch),
    }

    save_path = os.path.join(FINAL_MODEL_DIR, f"region_{int(region_id)}_OU_LSTM_regionspecific.pt")
    torch.save(model.state_dict(), save_path)

    artifact = {
        "df": dfr,
        "beta": beta,
        "arch": arch,
        "x_mean": x_mean,
        "x_std": x_std,
        "idx_t": idx_t,
        "train_mask": train_mask,
        "test_mask": test_mask,
        "epochs_logged": np.asarray(epochs_logged),
        "train_curve": np.asarray(train_curve),
        "test_curve": np.asarray(test_curve),
        "kappa_all": kappa_all,
        "sigma_all": sigma_all,
        "z_test": z_test,
        "metrics": metrics,
        "save_path": save_path,
    }
    return model, artifact


In [ ]:
# ============================================================
# TRAIN FINAL MODEL PER REGION
# ============================================================

final_models = {}
final_region_artifacts = {}
final_summary_rows = []

for r in GRID_REGIONS:
    print(f"Training final model for region {r}...")
    model_r, art_r = train_final_model_for_region(r)
    final_models[int(r)] = model_r
    final_region_artifacts[int(r)] = art_r

    final_summary_rows.append({
        "region": int(r),
        "hidden": art_r["arch"]["hidden"],
        "n_layers": art_r["arch"]["n_layers"],
        "head_hidden": art_r["arch"]["head_hidden"],
        "dropout": art_r["arch"]["dropout"],
        **art_r["metrics"],
    })

final_summary = pd.DataFrame(final_summary_rows).sort_values("region").reset_index(drop=True)
final_summary


In [ ]:
# ============================================================
# FINAL MODEL DIAGNOSTIC PLOTS FOR ONE REGION
# ============================================================

REGION_TO_PLOT = GRID_REGIONS[0]
art = final_region_artifacts[REGION_TO_PLOT]
dfr = art["df"]
dates_k = dfr.loc[art["idx_t"], "date"].values

plt.figure(figsize=(7, 4))
plt.plot(art["epochs_logged"], art["train_curve"], label="train")
plt.plot(art["epochs_logged"], art["test_curve"], label="test")
plt.title(f"Region {REGION_TO_PLOT}: NLL curves")
plt.xlabel("epoch")
plt.ylabel("NLL")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(dates_k, art["kappa_all"])
plt.title(f"Region {REGION_TO_PLOT}: learned kappa_t")
plt.xlabel("date")
plt.ylabel("kappa_t")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(dates_k, art["sigma_all"])
plt.title(f"Region {REGION_TO_PLOT}: learned sigma_t")
plt.xlabel("date")
plt.ylabel("sigma_t")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(art["z_test"], bins=30)
plt.title(f"Region {REGION_TO_PLOT}: standardized residuals (test)")
plt.xlabel("z")
plt.ylabel("count")
plt.show()


In [ ]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

final_summary[[
    "region", "hidden", "n_layers", "head_hidden", "dropout",
    "test_nll", "mae", "rmse", "r2", "corr",
    "picp_90", "coverage_error_90", "z_mean", "z_std", "z_skew", "z_kurtosis"
]]


In [ ]:
# ============================================================
# EXPORT OU-LSTM TEMPERATURE MODELS TO PKL
# ============================================================

CALIB_END = "2014-12-31"
OUT_DIR = Path("../Outputs/OULSTMParametersRegionSpecificModels/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "LSTM-OU Temperature Model (Region-Specific Architecture)"


def export_ou_lstm_models_to_pkls(regions_to_export, final_models, final_region_artifacts,
                                  calib_end=CALIB_END, out_dir=OUT_DIR, save_combined=True):
    region_models = {}

    for reg in regions_to_export:
        reg = int(reg)
        model = final_models[reg]
        art = final_region_artifacts[reg]
        payload = {
            "model_name": MODEL_NAME,
            "region": reg,
            "calibration_end": calib_end,
            "beta": np.asarray(art["beta"]),
            "x_mean": float(art["x_mean"]),
            "x_std": float(art["x_std"]),
            "arch": art["arch"],
            "metrics": art["metrics"],
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        }
        region_models[reg] = payload
        with open(out_dir / f"region_{reg}_ou_lstm_model.pkl", "wb") as f:
            pickle.dump(payload, f)

    if save_combined:
        with open(out_dir / "all_regions_ou_lstm_models.pkl", "wb") as f:
            pickle.dump(region_models, f)

    return region_models


exported_models = export_ou_lstm_models_to_pkls(GRID_REGIONS, final_models, final_region_artifacts)
print(f"Saved {len(exported_models)} region model files to {OUT_DIR}")
